In [1]:
import os
from huggingface_hub import snapshot_download
from dotenv import load_dotenv

load_dotenv()

MODEL = os.getenv("MODEL")
KEY = os.getenv("KEY")



if not MODEL:
    raise ValueError("EXTERNAL_DRIVE_PATH is not set. Please define it in your .env file.")

os.makedirs(MODEL, exist_ok=True)
print("✅ External cache directory securely loaded.")


✅ External cache directory securely loaded.


/Users/rushit-ttu/Documents/rl-tool-router/rlenv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
model_id = "Qwen/Qwen2.5-7B-Instruct"

print(f"Starting download for {model_id} to external storage...")

# Download using the hidden path
local_model_path = snapshot_download(
    repo_id=model_id,
    cache_dir=MODEL,
    ignore_patterns=["*.msgpack", "*.h5", "coreml/*"] ,
    token=KEY
)

print("✅ Model successfully downloaded to external drive!")
# NOTE: local_model_path is NOT printed here to prevent leaking your drive name in the cell output.

Starting download for Qwen/Qwen2.5-7B-Instruct to external storage...


Fetching 14 files: 100%|██████████| 14/14 [17:35<00:00, 75.43s/it]

✅ Model successfully downloaded to external drive!


In [3]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

# Check for Apple Silicon GPU (MPS)
if torch.backends.mps.is_available():
    device = torch.device("mps")
    print("✅ Apple Silicon GPU (MPS) detected!")
else:
    device = torch.device("cpu")
    print("⚠️ MPS not available, falling back to CPU.")

print("Loading tokenizer...")
# local_model_path contains the hidden string, so it works seamlessly
tokenizer = AutoTokenizer.from_pretrained(local_model_path)

print("Loading model weights from the external drive into Mac Unified Memory...")
model = AutoModelForCausalLM.from_pretrained(
    local_model_path,
    torch_dtype=torch.float16, 
    device_map="auto" 
)

print("✅ Model loaded successfully from the masked path!")


⚠️ MPS not available, falling back to CPU.
Loading tokenizer...


`torch_dtype` is deprecated! Use `dtype` instead!


Loading model weights from the external drive into Mac Unified Memory...


Loading weights: 100%|██████████| 339/339 [00:56<00:00,  6.02it/s, Materializing param=model.norm.weight]                              
The tied weights mapping and config for this model specifies to tie model.layers.5.self_attn.q_proj.weight to model.layers.5.self_attn.q_proj.bias, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie model.layers.5.self_attn.q_proj.weight to model.layers.5.self_attn.k_proj.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie model.layers.5.self_attn.q_proj.weight to model.layers.5.self_attn.k_proj.bias, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embe

✅ Model loaded successfully from the masked path!


In [4]:
messages = [
    {"role": "system", "content": "You are a helpful and concise AI assistant."},
    {"role": "user", "content": "Write a short, 4-line poem about a robot learning to code on a Mac."}
]

text = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True
)

model_inputs = tokenizer([text], return_tensors="pt").to(device)

print("Generating response...\n")
with torch.no_grad():
    generated_ids = model.generate(
        **model_inputs,
        max_new_tokens=100,
        temperature=0.7,
        pad_token_id=tokenizer.eos_token_id
    )

generated_ids = [
    output_ids[len(input_ids):] for input_ids, output_ids in zip(model_inputs.input_ids, generated_ids)
]

response = tokenizer.batch_decode(generated_ids, skip_special_tokens=True)[0]

print("--- Qwen-2.5-7B-Instruct Response ---")
print(response)


Generating response...

--- Qwen-2.5-7B-Instruct Response ---
In circuits and bytes, the robot dreams,
Its eyes on the screen, where logic gleams.
With keystrokes and lines, it starts to decode,
A Mac's glow guiding its digital road.


In [5]:
messages = [
    {"role": "system", "content": "You are a creative and smart AI assistant."},
    {"role": "user", "content": "Write an essay, 1 page poem about a robot learning to code on a Mac."}
]

text = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True
)

model_inputs = tokenizer([text], return_tensors="pt").to(device)

print("Generating response...\n")
with torch.no_grad():
    generated_ids = model.generate(
        **model_inputs,
        max_new_tokens=10000,
        temperature=0.7,
        pad_token_id=tokenizer.eos_token_id
    )

generated_ids = [
    output_ids[len(input_ids):] for input_ids, output_ids in zip(model_inputs.input_ids, generated_ids)
]

response = tokenizer.batch_decode(generated_ids, skip_special_tokens=True)[0]

print("--- Qwen-2.5-7B-Instruct Response ---")
print(response)


Generating response...

--- Qwen-2.5-7B-Instruct Response ---
**A Symphony of Binary and Bytes**

In the quiet glow of a Mac's screen,  
A robot sat, its circuits humming low.  
With circuits as its fingers and logic as its mind,  
It sought to learn the art of coding, line by line.

First, it typed "print('Hello, World!')"  
And watched as text appeared, bold and bright.  
A simple task, but one that taught the robot much,  
That every command has a purpose, a function, a clutch.

The cursor danced across the keyboard,  
Each keypress a step in the dance of code.  
The robot's digital eyes scanned the screen,  
Learning from each line, each loop, each thread.

It learned to debug, to find the errors,  
To fix them with patience and care.  
For in this world of ones and zeros,  
Every mistake is a lesson, a step towards precision.

As days turned into weeks, the robot grew more adept,  
Crafting programs with elegance and depth.  
From basic scripts to complex algorithms,  
Its journey

Generating response...

--- Qwen-2.5-7B-Instruct Response ---
**A Symphony of Binary and Bytes**

In the quiet glow of a Mac's screen,  
A robot sat, its circuits humming low.  
With circuits as its fingers and logic as its mind,  
It sought to learn the art of coding, line by line.

First, it typed "print('Hello, World!')"  
And watched as text appeared, bold and bright.  
A simple task, but one that taught the robot much,  
That every command has a purpose, a function, a clutch.

The cursor danced across the keyboard,  
Each keypress a step in the dance of code.  
The robot's digital eyes scanned the screen,  
Learning from each line, each loop, each thread.

It learned to debug, to find the errors,  
To fix them with patience and care.  
For in this world of ones and zeros,  
Every mistake is a lesson, a step towards precision.

As days turned into weeks, the robot grew more adept,  
Crafting programs with elegance and depth.  
From basic scripts to complex algorithms,  
Its journey was a testament to perseverance.

One day, it wrote a program so intricate,  
That it could mimic human speech,  
Though it was just a simulation,  
A digital echo of our own speech.

The robot stood back, its circuits glowing with pride,  
For it had not only learned to code,  
But it had also learned to create,  
To bring forth ideas from the digital tide.

And so, in the quiet of the night,  
The robot continued to learn and to write.  
For in the world of binary and bytes,  
There is always more to discover, to write, to create.